In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ضبط تخفيف الأخطاء

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** إصدار بيتا لنموذج تنفيذ جديد متاح الآن. يوفر نموذج التنفيذ الموجَّه مرونة أكبر عند تخصيص سير عمل تخفيف الأخطاء. راجع دليل [نموذج التنفيذ الموجَّه](/guides/directed-execution-model) للمزيد من المعلومات.


<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو ما هو أحدث منها.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
تتيح تقنيات تخفيف الأخطاء للمستخدمين التخفيف من أخطاء الدوائر الكمومية عن طريق
نمذجة ضوضاء الجهاز في وقت التنفيذ. يؤدي هذا عادةً إلى
تكاليف معالجة كمومية مسبقة مرتبطة بتدريب النموذج،
وتكاليف معالجة كلاسيكية لاحقة لتخفيف الأخطاء في النتائج الخام
باستخدام النموذج المولَّد.

يدعم الـ Estimator primitive عدة تقنيات لتخفيف الأخطاء، من بينها [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)، و[ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)، و[PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)، و[PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). راجع [تقنيات تخفيف الأخطاء وقمعها](error-mitigation-and-suppression-techniques) للاطلاع على شرح كل منها. عند استخدام الـ primitives، يمكنك تشغيل أو إيقاف الطرق الفردية. راجع قسم [إعدادات الأخطاء المخصصة](#advanced-error) لمزيد من التفاصيل.

> **Note:** لا يدعم Sampler تخفيف الأخطاء، لكن يمكنك استخدام حزمة [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (تخفيف قياس خالٍ من المصفوفات) لتنفيذ تخفيف الأخطاء محلياً.

يدعم Estimator أيضاً `resilience_level`. يحدد مستوى المرونة مقدار الصمود ضد
الأخطاء. تولّد المستويات الأعلى نتائج أكثر دقة، على حساب
أوقات معالجة أطول. يمكن استخدام مستويات المرونة لضبط
التوازن بين التكلفة والدقة عند تطبيق تخفيف الأخطاء على استعلام الـ primitive.
يقلل تخفيف الأخطاء من الأخطاء (التحيز) في النتائج عن طريق معالجة
مخرجات مجموعة، أو طيف، من الدوائر ذات الصلة. تعتمد
درجة تقليل الأخطاء على الطريقة المطبَّقة. يُجرِّد مستوى المرونة
الاختيار التفصيلي لطريقة تخفيف الأخطاء ليتيح
للمستخدمين التفكير في التوازن بين التكلفة والدقة المناسب
لتطبيقهم.

بناءً على ذلك، يقابل كل مستوى طريقةً أو طرقاً تتزايد معها
تكاليف أخذ العينات الكمومية، لتتمكن من تجربة
توازنات مختلفة بين الوقت والدقة. يوضح الجدول التالي
المستويات والطرق المقابلة المتاحة لكل من
الـ primitives.

> **Info:** تخفيف الأخطاء مرتبط بالمهمة المحددة، لذا تتفاوت التقنيات التي يمكنك
> تطبيقها بحسب ما إذا كنت تأخذ عينات من توزيع أو تولّد
> قيم توقع.

<span id="resilience-table"></span>

يدعم Estimator مستويات المرونة التالية. لا يدعم Sampler مستويات المرونة.

| مستوى المرونة | التعريف                                                                                            | التقنية                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | بلا تخفيف                                                                                         | لا شيء                                                                      |
| 1 [الافتراضي]      | أدنى تكاليف تخفيف: تخفيف الأخطاء المرتبطة بأخطاء القراءة               | إطفاء أخطاء القراءة المُلتوي (TREX) مع التلوية في القياس             |
| 2                | تكاليف تخفيف متوسطة. يقلل عادةً من التحيز في المُقدِّرات، لكنه غير مضمون لإنتاج نتائج خالية من التحيز. | المستوى 1 + الاستقراء عند انعدام الضوضاء (ZNE) مع تلوية البوابات

> **Info:** مستويات المرونة في مرحلة بيتا حالياً، لذا ستتباين تكاليف أخذ العينات و
> جودة الحل من دائرة إلى أخرى. ستُطلَق ميزات جديدة،
> وخيارات متقدمة، وأدوات إدارة بشكل متدرج. لا يُضمن تطبيق
> طرق تخفيف أخطاء محددة عند كل مستوى مرونة.

## ضبط Estimator بمستويات المرونة
يمكنك استخدام مستويات المرونة لتحديد تقنيات تخفيف الأخطاء، أو يمكنك ضبط التقنيات المخصصة بشكل فردي كما هو موضح في [إعدادات الأخطاء المخصصة.](#advanced-error)

<details>
<summary>مستوى المرونة 0</summary>

لا يُطبَّق أي تخفيف للأخطاء على برنامج المستخدم.

</details>

<details>
<summary>مستوى المرونة 1</summary>

يطبّق المستوى 1 **تخفيف أخطاء القراءة** و**تلوية القياس** عبر تقنية خالية من النماذج تُعرَف
بـ إطفاء أخطاء القراءة المُلتوي (TREX). يقلل هذا من أخطاء القياس
عن طريق تشخيص قناة الضوضاء المرتبطة بالقياس من خلال
قلب البتات الكمومية Qubits بشكل عشوائي عبر بوابات X مباشرةً قبل القياس. يتم تعلّم
عامل إعادة الضبط من قناة الضوضاء القطرية
عبر قياس دوائر عشوائية مُهيَّأة في الحالة الصفرية. يتيح هذا
للخدمة إزالة التحيز من قيم التوقع الناتجة عن
ضوضاء القراءة. هذا النهج موضح بمزيد من التفصيل في [تخفيف أخطاء القراءة الخالي من النماذج لقيم التوقع الكمومي](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>مستوى المرونة 2</summary>

يطبّق المستوى 2 **تقنيات تخفيف الأخطاء المدرجة في المستوى 1** ويطبّق أيضاً **تلوية البوابات** ويستخدم **طريقة الاستقراء عند انعدام الضوضاء (ZNE)**. تحسب ZNE
قيمة التوقع للمتغيّر الملاحَظ لعوامل ضوضاء مختلفة
(مرحلة التضخيم) ثم تستخدم قيم التوقع المقاسة
للاستنتاج بقيمة التوقع المثالية عند حد انعدام الضوضاء (مرحلة الاستقراء). يميل هذا النهج إلى تقليل الأخطاء في قيم التوقع، لكنه
غير مضمون لإنتاج نتيجة خالية من التحيز.

![تُظهر هذه الصورة رسماً بيانياً. المحور السيني مُعنوَن بـ عامل تضخيم الضوضاء. المحور الصادي مُعنوَن بـ قيمة التوقع. خط مائل للأعلى مُعنوَن بـ القيمة المخففة. نقاط قريبة من الخط هي قيم مُضخَّمة بالضوضاء. هناك خط أفقي فوق محور السينات مباشرةً مُعنوَن بـ القيمة الدقيقة.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "توضيح طريقة ZNE")

تتناسب تكلفة هذه الطريقة مع عدد عوامل الضوضاء. تأخذ
الإعدادات الافتراضية عينات من قيمة التوقع عند ثلاثة عوامل ضوضاء،
مما يؤدي إلى تكلفة أعلى بنحو 3 أضعاف عند استخدام مستوى المرونة هذا.

في المستوى 2، تقوم طريقة TREX بقلب البتات الكمومية Qubits بشكل عشوائي عبر بوابات X مباشرةً قبل القياس،
وتقلب البت المقاس المقابل إذا طُبِّقت بوابة X. هذا النهج موضح بمزيد من التفصيل في [تخفيف أخطاء القراءة الخالي من النماذج لقيم التوقع الكمومي](https://arxiv.org/abs/2012.09738).

</details>

### مثال
تتيح واجهة `EstimatorV2` للمستخدمين العمل بسلاسة مع مجموعة متنوعة من
طرق تخفيف الأخطاء لتقليل الأخطاء في قيم التوقع للمتغيرات الملاحَظة. يستخدم الكود التالي الاستقراء عند انعدام الضوضاء وتخفيف أخطاء القراءة ببساطة
عبر تعيين `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## إعدادات الأخطاء المخصصة
يمكنك تشغيل أو إيقاف طرق تخفيف الأخطاء وقمعها بشكل فردي، بما فيها الفصل الديناميكي، وتلوية البوابات والقياس، وتخفيف أخطاء القياس، و PEC، و ZNE. راجع [تقنيات تخفيف الأخطاء وقمعها](error-mitigation-and-suppression-techniques) للاطلاع على شرح كل منها.

> **Note:** - لا تتوفر جميع الخيارات للـ primitives كليهما.   راجع [جدول الخيارات المتاحة](runtime-options-overview#options-table) للحصول على قائمة الخيارات المتاحة.
> - لا تعمل جميع الطرق معاً على جميع أنواع الدوائر.  راجع [جدول توافق الميزات](runtime-options-overview#options-compatibility-table) للمزيد من التفاصيل.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">